In [1]:
from filtering_utils import (
    SYSTEM_PROMPT, get_statement, StructuredExtractionOutput,
    STRUCTURED_EXTRACTION_SCHEMA, _char_to_token_idx, apply_correction, apply_manual_corrections
)
from pathlib import Path
import os
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv
import time
import datetime 
import json
import pandas as pd
from ast import literal_eval
from tqdm import tqdm
from transformers import AutoTokenizer

### Filtering A1_train

In [4]:
A1_train = pd.read_csv("../CoT_datasets/raw/A1_train.csv")
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")

##### Creating the Request File

The Azure `o3` deployment is `GlobalBatch`-only — no real-time calls, so `evaluate_sample`/`filter_dataset` above can't be used against it. Instead we build a `.jsonl` request file, submit it as a batch job, poll until it finishes, then parse the results back into the same `extraction_point_is_correct`/`new_pre_final_judgement_phrase` shape.

This cell only builds the input `.jsonl` file for A1_train — submission/polling/parsing come after.

Two things to verify before submitting, that I'm not 100% certain of for this specific deployment:
- `url: "/chat/completions"` is the standard field value for both OpenAI's and Azure's Batch API — check this still holds for a GlobalBatch deployment exposed through the newer `/openai/v1/` surface.
- `o3` is a reasoning model; reasoning models generally reject `temperature` as a sampling parameter (unlike GPT-4.1, which OpenRouter was using it with) — omitted here rather than guessed wrong.

In [ ]:

AZURE_BATCH_DIR = Path("azure_batches")
AZURE_BATCH_DIR.mkdir(exist_ok=True)

AZURE_DEPLOYMENT_NAME = "o3"

STRUCTURED_EXTRACTION_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "StructuredExtractionOutput",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "extraction_point_is_correct": {"type": "boolean"},
                "new_pre_final_judgement_phrase": {"type": ["string", "null"]},
            },
            "required": ["extraction_point_is_correct", "new_pre_final_judgement_phrase"],
            "additionalProperties": False,
        },
    },
}


def build_batch_request(custom_id: str, sample: pd.Series) -> dict:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",  # fixed: needs the /v1/ prefix per the docs
        "body": {
            "model": AZURE_DEPLOYMENT_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"""Statement: {get_statement(sample["generated_statement_texts"])}

    LLM Generated Output:
    {sample["generated_statement_texts"]}

    Extracted Text:
    {sample["extracted_statement_texts"]}

    Determine whether the extraction is correct.""",
                },
            ],
            "response_format": STRUCTURED_EXTRACTION_SCHEMA,
        },
    }


def write_batch_jsonl(dataset: pd.DataFrame, task: str, split: str) -> Path:
    out_path = AZURE_BATCH_DIR / f"{task}_{split}_batch_input.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for idx, row in dataset.iterrows():
            custom_id = f"{task}_{split}_{idx}"
            request = build_batch_request(custom_id, row)
            line = json.dumps(request, ensure_ascii=False)
            f.write(line)
            f.write("\n")   # written as a separate call, so it can't get silently dropped
    print(f"Wrote {len(dataset)} requests to {out_path}")
    return out_path


a1_train_batch_path = write_batch_jsonl(A1_train, task="A1", split="train")


Wrote 700 requests to azure_batches\A1_train_batch_input.jsonl


##### Upload Batch File

In [ ]:
load_dotenv(dotenv_path="azure-openai.env")
    
client = OpenAI(
    base_url = "https://student-hub-0122.openai.azure.com/openai/v1",  
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
    )

# Upload a file with a purpose of "batch"
file = client.files.create(
  file=open("azure_batches/A1_train_batch_input.jsonl", "rb"), 
  purpose="batch",
  extra_body={"expires_after":{"seconds": 1209600, "anchor": "created_at"}} # Optional you can set to a number between 1209600-2592000. This is equivalent to 14-30 days
)

print(file.model_dump_json(indent=2))

print(f"File expiration: {datetime.fromtimestamp(file.expires_at) if file.expires_at is not None else 'Not set'}")

file_id = file.id

{
  "id": "file-0b89cb144dae4025bcf1ba5c0516810d",
  "bytes": 13243694,
  "created_at": 1787163638,
  "filename": "A1_train_batch_input.jsonl",
  "object": "file",
  "purpose": "batch",
  "status": "processed",
  "expires_at": 1788373238,
  "status_details": null
}
File expiration: 2026-09-02 21:20:38


##### Creating Batch Job

In [ ]:
# Submit a batch job with the file
batch_response = client.batches.create(
    input_file_id=file_id,
    endpoint="/chat/completions", # While passing this parameter is required, the system will read your input file to determine if the chat completions or responses API is needed.  
    completion_window="24h",
    # extra_body={"output_expires_after":{"seconds": 1209600, "anchor": "created_at"}} # Optional you can set to a number between 1209600-2592000. This is equivalent to 14-30 days
)

# Save batch ID for later use
batch_id = batch_response.id

print(batch_response.model_dump_json(indent=2))

{
  "id": "batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed",
  "completion_window": "24h",
  "created_at": 1787163642,
  "endpoint": "",
  "input_file_id": "file-0b89cb144dae4025bcf1ba5c0516810d",
  "object": "batch",
  "status": "validating",
  "cancelled_at": null,
  "cancelling_at": null,
  "completed_at": null,
  "error_file_id": "",
  "errors": null,
  "expired_at": null,
  "expires_at": null,
  "failed_at": null,
  "finalizing_at": null,
  "in_progress_at": null,
  "metadata": null,
  "model": null,
  "output_file_id": "",
  "request_counts": {
    "completed": 0,
    "failed": 0,
    "total": 0
  },
  "usage": null,
  "priority": "high"
}


##### Tracking Progress

In [ ]:
status = "validating"
while status not in ("completed", "failed", "canceled"):
    time.sleep(60)
    batch_response = client.batches.retrieve(batch_id)
    status = batch_response.status
    print(f"{datetime.datetime.now()} Batch Id: {batch_id},  Status: {status}")

if batch_response.status == "failed":
    for error in batch_response.errors.data:  
        print(f"Error code {error.code} Message {error.message}")

2026-08-19 21:21:52.878587 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: validating
2026-08-19 21:22:53.354409 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: in_progress
2026-08-19 21:23:53.866568 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: in_progress
2026-08-19 21:24:54.379442 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: in_progress
2026-08-19 21:25:54.874825 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: in_progress
2026-08-19 21:27:02.307578 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: in_progress
2026-08-19 21:28:02.711239 Batch Id: batch_cb7ea1a0-935a-4cdd-b2ac-8af92f7706ed,  Status: completed


##### Retreive Output File

In [ ]:
responses = []
output_file_id = batch_response.output_file_id

if not output_file_id:
    output_file_id = batch_response.error_file_id

if output_file_id:
    file_response = client.files.content(output_file_id)
    raw_responses = file_response.text.strip().split('\n')  

    for raw_response in raw_responses:  
        json_response = json.loads(raw_response)  
        responses.append(json_response)
        # formatted_json = json.dumps(json_response, indent=2)  
        # print(formatted_json)

In [ ]:


def apply_batch_results(dataset, batch_responses, tokenizer, dataset_file_path, task, split):
    if len(dataset) > 0 and isinstance(dataset["generated_statement_ids"].iloc[0], str):
        dataset["generated_statement_ids"] = dataset["generated_statement_ids"].apply(literal_eval)

    prefix = f"{task}_{split}_"
    n_applied = n_errors = 0

    for item in tqdm(batch_responses):
        custom_id = item["custom_id"]
        if not custom_id.startswith(prefix):
            continue
        idx = int(custom_id[len(prefix):])

        if item.get("error") is not None:
            tqdm.write(f"Row {idx}: batch request errored: {item['error']}")
            n_errors += 1
            continue

        status_code = item["response"].get("status_code")
        if status_code != 200:
            tqdm.write(f"Row {idx}: non-200 status ({status_code}), skipping")
            n_errors += 1
            continue

        try:
            correction = json.loads(item["response"]["body"]["choices"][0]["message"]["content"])
        except Exception as e:
            tqdm.write(f"Row {idx}: couldn't parse structured output ({e!r}), skipping")
            n_errors += 1
            continue

        if apply_correction(dataset, idx, dataset.loc[idx], correction, tokenizer):
            n_applied += 1

    dataset.to_csv(dataset_file_path, index=False)
    print(f"Applied {n_applied} corrections, {n_errors} errors/skips, out of {len(batch_responses)} batch responses")
    return dataset


In [ ]:
filtered_A1_train = apply_batch_results(
    dataset=A1_train, batch_responses=responses, tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/A1_filtered_train.csv",
    task="A1", split="train",
)

  7%|▋         | 48/700 [00:00<00:03, 210.93it/s]

Row 19: phrase not found verbatim, skipping: 'Wait, that doesn’t'
Row 85: phrase not found verbatim, skipping: '43 as stated in the problem. Wait, that doesn\x19t'
Row 81: phrase found multiple times (ambiguous), skipping: 'equation is'
Row 78: phrase not found verbatim, skipping: 'answer says -29. That doesn\x19t'


 13%|█▎        | 91/700 [00:00<00:03, 175.26it/s]

Row 100: phrase not found verbatim, skipping: 'The statement is'


 24%|██▍       | 167/700 [00:00<00:02, 181.22it/s]

...

Row 602: phrase not found verbatim, skipping: 'Therefore, the statement is'
Row 654: phrase not found verbatim, skipping: 'says 41*7=282. That doesn\x0039t'


 96%|█████████▌| 672/700 [00:04<00:00, 165.68it/s]

Row 658: phrase found multiple times (ambiguous), skipping: "Wait, that doesn't"
Row 663: phrase not found verbatim, skipping: 'equation, which is 7140. That'


100%|██████████| 700/700 [00:04<00:00, 166.62it/s]


Row 670: phrase not found verbatim, skipping: 'That doesn’t'
Applied 470 corrections, 0 errors/skips, out of 700 batch responses


In [5]:
# A1_filtered_train manual corrections - 29 filter-failed rows (24 fixed + 5 drops).
# A1 = simplest equation verification 'A op B = result'. Keep computation, cut at comparison to RHS.
# nomatch rows land on 'That'/'that' (drop negation) per manual standard. 5 DROPS = truncated (no </think>).

corrections = {
    19: 'total to 28. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    78: 'answer says -29. That',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    81: 'stated in the equation is',  # explicit verdict 'the (result stated in the) equation/statement is wrong'; land copula, excl verdict.
    85: 'the problem. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    100: 'drop',  # DROP - truncated, no </think>
    131: 'gives 54. That',  # computed LHS matches RHS: 'That matches'; land 'That', excl 'matches'. Computed value kept.
    149: 'drop',  # DROP - truncated, no </think>
    152: 'says 2*68=140. Hmm, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    180: 'equals 7290. That',  # computed LHS matches RHS: 'That matches'; land 'That', excl 'matches'. Computed value kept.
    194: 'drop',  # DROP - truncated, no </think>
    204: 'equal to 36. That',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    206: 'drop',  # DROP - truncated, no </think>
    207: 'gives me 78. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    262: 'equals 1377. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    304: 'with -$31. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    339: 'not -10. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    395: 'gives us 7120. That',  # computed LHS matches RHS: 'That matches'; land 'That', excl 'matches'. Computed value kept.
    404: 'gives me 107. Hmm, wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    418: 'be -26. That',  # computed LHS matches RHS: 'That matches'; land 'That', excl 'matches'. Computed value kept.
    462: 'equals 92. That',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    499: 'equals 111. That',  # no explicit match-word; first affirmation 'That seems right' (re: the computed value); land 'That', excl 'seems right'.
    551: 'equal 170. That',  # computed LHS matches RHS: 'That matches'; land 'That', excl 'matches'. Computed value kept.
    566: 'claiming it equals 139 is',  # explicit verdict 'the (result stated in the) equation/statement is wrong'; land copula, excl verdict.
    589: 'equals 67. That',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    602: 'drop',  # DROP - truncated, no </think>
    654: 'says 41*7=282. That',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    658: 'land at -27. Wait, that',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
    663: '= 7140. That',  # computed LHS matches RHS: 'That matches'; land 'That', excl 'matches'. Computed value kept.
    670: '87 = 0. That',  # LHS != RHS: informal 'That/that doesn't match/seem right'; land the pronoun, excl 'doesn't...' (manual standard = drop negation; the o3-vs-manual convention still open).
}

filtered_A1_train = apply_manual_corrections(
    dataset=filtered_A1_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/A1_filtered_train.csv",
)

Row 19: corrected.
Row 78: corrected.
Row 81: corrected.
Row 85: corrected.
Row 131: corrected.
Row 152: corrected.
Row 180: corrected.
Row 204: corrected.
Row 207: corrected.
Row 262: corrected.
Row 304: corrected.
Row 339: corrected.
Row 395: corrected.
Row 404: corrected.
Row 418: corrected.
Row 462: corrected.
Row 499: corrected.
Row 551: corrected.
Row 566: corrected.
Row 589: corrected.
Row 654: corrected.
Row 658: corrected.
Row 663: corrected.
Row 670: corrected.
Dropping 5 row(s): [100, 149, 194, 206, 602]
